In [1]:
%matplotlib osx 
## Needed to make the plots work on MacOS otherwise it will show inline in the notebook.
import configparser
from tqdm.notebook import tqdm
import os
import numpy as np
import pylidc as pl
import matplotlib, matplotlib.pyplot as plt
plt.cm.cmap_d = {name: matplotlib.colormaps[name] for name in matplotlib.colormaps}

configparser.SafeConfigParser = configparser.ConfigParser
np.int, np.float, np.bool = int, float, bool

/opt/miniconda3/lib/python3.14/site-packages/pylidc/__init__.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as _pr


# Getting to know the data & pylidc library
Dataset used: https://www.cancerimagingarchive.net/collection/lidc-idri/
They advice the use of `pydilc` for the preprocessing and conversions of the data.

**Important!: code on the pydilc site is not up-to-date with the real implementation**_

You can print the current implementation of the function `print(inspect.getsource(pl.Scan.###))`
 

In [2]:
print(inspect.getsource(pl.Scan.load_all_dicom_images))

NameError: name 'inspect' is not defined

In [ ]:
print(inspect.getsource(pl.Scan.to_volume))

    def to_volume(self, verbose=True):
        """
        Return the scan as a 3D numpy array volume.
        """
        images = self.load_all_dicom_images(verbose=verbose)

        volume = np.stack(
            [
                x.pixel_array * x.RescaleSlope + x.RescaleIntercept
                for x in images
            ],
            axis=-1,
        ).astype(np.int16)
        return volume



Out of this we can conclude that HU-conversion already happens under the hood in the `pylidc` library when calling `to_volume()` on a scan.

## Loading one patient data

In [ ]:
pid = 'LIDC-IDRI-0039'
scan = pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).first() ## Different possible filters (see: https://pylidc.github.io/scan.html#pylidc.Scan). .first() returns the first scan that matches the filter.
print(len(scan.annotations)) ## Show the amount of annotations for this scan

## To cluster the different annotations into nodules, we can use the cluster_annotations() method. This will return a list of lists, where each inner list contains the annotations that belong to the same nodule.
nodules = scan.cluster_annotations()
print("Nodules list:", nodules)
for i,nod in enumerate(nodules):
    print("Nodule %d has %d annotations." % (i+1, len(nodules[i])))

#scan.visualize(annotation_groups=nodules)

# To convert image scan to numpy array, we can use the to_volume() method. 
volume = scan.to_volume(verbose=False)          
print("Volume min:", volume.min())
print("Volume max:", volume.max())
print("Volume shape:", volume.shape) ## The shape of the volume is (height, width, slices)
## So 'slices' amount of CT images of size (height, width) are stacked together to form the volume. The slices are ordered from top to bottom of the body. The pixel values are in Hounsfield units (HU).

20
Nodules list: [[Annotation(id=356,scan_id=50), Annotation(id=365,scan_id=50), Annotation(id=370,scan_id=50), Annotation(id=371,scan_id=50)], [Annotation(id=357,scan_id=50), Annotation(id=364,scan_id=50), Annotation(id=369,scan_id=50), Annotation(id=373,scan_id=50)], [Annotation(id=358,scan_id=50), Annotation(id=362,scan_id=50), Annotation(id=368,scan_id=50), Annotation(id=374,scan_id=50)], [Annotation(id=359,scan_id=50), Annotation(id=363,scan_id=50), Annotation(id=367,scan_id=50), Annotation(id=372,scan_id=50)], [Annotation(id=360,scan_id=50), Annotation(id=361,scan_id=50), Annotation(id=366,scan_id=50), Annotation(id=375,scan_id=50)]]
Nodule 1 has 4 annotations.
Nodule 2 has 4 annotations.
Nodule 3 has 4 annotations.
Nodule 4 has 4 annotations.
Nodule 5 has 4 annotations.
Volume min: -3024
Volume max: 3071
Volume shape: (512, 512, 281)


## Annotations

In [ ]:
# Getting one annotations
ann = pl.query(pl.Annotation)\
        .filter(pl.Annotation.spiculation > 3).first() ## Get the first annotation of the first nodule
ann.print_formatted_feature_table()
## Plot the annotation in 3D and in scan view.
#ann.visualize_in_3d(edgecolor='green', cmap='autumn')
## Plot the annotation in scan view. The scan view shows the CT images of the scan with the annotation overlaid in red on top.
#ann.visualize_in_scan()

Feature              Meaning                    # 
-                    -                          - 
Subtlety           | Obvious                  | 5 
Internalstructure  | Soft Tissue              | 1 
Calcification      | Non-central              | 4 
Sphericity         | Ovoid/Round              | 4 
Margin             | Near Sharp               | 4 
Lobulation         | Near Marked Lobulation   | 4 
Spiculation        | Near Marked Spiculation  | 4 
Texture            | Solid                    | 5 
Malignancy         | Highly Suspicious        | 5 
Loading dicom files ... This may take a moment.


# Preprocessing of data


In [ ]:
def process_scan(scan):
    ## Convert the scan to a 3D volume
    vol = scan.to_volume(verbose=False)          # (x, y, z (slices)) already in HU

    ## Clipping to make the parts of interest more visible. So values of hu_max are a 1 (white) a nd values of hu_min are a 0 (black). The values in between are scaled linearly.
    # Healthy lung tissue has a Hounsfield unit (HU) value of around -950 to -650 (https://radiopaedia.org/articles/hounsfield-unit)
    # However we also wan't to include the nodules which typically have a higher HU value.
    hu_min = -1000
    hu_max = 400
    vol = np.clip(vol, hu_min, hu_max).astype(np.float32)

    
    # 1. resamplen naar 1x1x1 mm
    # 2. clippen + normaliseren
    # 3. noduli clusteren en consensus bepalen
    # 4. positieve patches uitknippen en opslaan
    # 5. negatieve patches trekken en opslaan
    # 6. rijen voor de CSV teruggeven
    return rows

all_rows = []

scans = pl.query(pl.Scan).all() 
for scan in tqdm(scans):
    try:
        print(scan.patient_id)
        volume = scan.to_volume(verbose=False)
        print("Volume shape:", volume.shape)

    except Exception as e:
        print("Mislukt:", scan.patient_id, e)

  0%|          | 0/1018 [00:00<?, ?it/s]

LIDC-IDRI-0078
Volume shape: (512, 512, 87)
LIDC-IDRI-0069
Volume shape: (512, 512, 136)
LIDC-IDRI-0079
Volume shape: (512, 512, 113)
LIDC-IDRI-0101
Volume shape: (512, 512, 150)
LIDC-IDRI-0110
Volume shape: (512, 512, 125)
LIDC-IDRI-0115
Volume shape: (512, 512, 112)
LIDC-IDRI-0132
Volume shape: (512, 512, 127)
LIDC-IDRI-0136
Volume shape: (512, 512, 153)
LIDC-IDRI-0150
Volume shape: (512, 512, 102)
LIDC-IDRI-0151
Volume shape: (512, 512, 111)
LIDC-IDRI-0154
Volume shape: (512, 512, 115)
LIDC-IDRI-0001
Volume shape: (512, 512, 133)
LIDC-IDRI-0002


KeyboardInterrupt: 

# _Tests on data_

### _*1:  Is the voxel size different?_

In [6]:
scans = pl.query(pl.Scan).all()
for i in tqdm(range(5)):
    scan = scans[i]

    print("\nPatient:", scan.patient_id)

    volume = scan.to_volume(verbose=False)

    print("Volume shape:", volume.shape)
    print("Pixel spacing:", scan.pixel_spacing)
    print("Slice thickness:", scan.slice_thickness)

  0%|          | 0/5 [00:00<?, ?it/s]


Patient: LIDC-IDRI-0078
Volume shape: (512, 512, 87)
Pixel spacing: 0.65
Slice thickness: 3.0

Patient: LIDC-IDRI-0069
Volume shape: (512, 512, 136)
Pixel spacing: 0.741
Slice thickness: 2.0

Patient: LIDC-IDRI-0079
Volume shape: (512, 512, 113)
Pixel spacing: 0.698
Slice thickness: 3.0

Patient: LIDC-IDRI-0101
Volume shape: (512, 512, 150)
Pixel spacing: 0.626953125
Slice thickness: 3.0

Patient: LIDC-IDRI-0110
Volume shape: (512, 512, 125)
Pixel spacing: 0.73046875
Slice thickness: 3.0


Conclusion: yes voxel size differs. Likely because of the different origins (different scanners from different hospitals). 

Clear image to illustrate a voxel.and it's sice: https://pmc.ncbi.nlm.nih.gov/articles/PMC11202317/#:~:text=Fig%2E%201,-%2E
A voxel is like a 2D pixel but then in 3D. It is characterized by a Size = (Δx, Δy, Δz). Where:
* Δx = pixel spacing in the horizontal direction
* Δy = pixel spacing in the vertical direction
* Δz = slice thickness (or slice spacing, depending on the image metadata)

To ensure consistent spatial dimensions across all CT scans, we resample the volumes to a uniform voxel spacing of 1 × 1 × 1 mm³.
This allows the AI model to process anatomical structures at a consistent physical scale, reducing variations caused by differences in voxel spacing between scans.